# EXP-001 · Qué soportan de verdad los datos sobre un extent

## 1 · Para qué sirve este notebook y cómo se usa

Antes de construir ningún pipeline hay que saber qué entregan **realmente** los servicios
de ortofoto, catastro y elevación. Este notebook lo mide sobre un único rectángulo del
mapa y responde cuatro preguntas — o dice por qué no ha podido. Es el instrumento de
[EXP-001](../docs/experiments/exp-001-data-feasibility.md).

| # | Pregunta | La miden las secciones |
|---|---|---|
| 1 | ¿A qué resolución y en qué años cubre la ortofoto el extent? | 5 y 6 |
| 2 | ¿Qué productos de elevación lo cubren, y en qué forma? | 9 |
| 3 | ¿Qué indicadores de abandono se pueden calcular con eso? | 7 y 12 |
| 4 | ¿Se pueden etiquetar a mano suficientes ejemplos? ¿Ayuda el catastro? | 8, 10 y 11 |

Se ejecuta con dos comandos, desde la raíz del repositorio:

```bash
uv run --group eda python notebooks/exp-001/eda.py --selftest   # comprobaciones sin red
uv run --group eda jupyter lab notebooks/exp-001/01-eda-pnoa-catastro.ipynb
```

El experimento es **autocontenido**: todo vive en esta carpeta, datos incluidos, así que
se puede mover entera sin romper nada. Lo que escribe cae en `out/`, junto a este
notebook: `manifest.csv` —una línea por petición, con URL, estado, bytes y MD5—,
`extent.json`, `drawing.json`, `candidates.csv`, `summary.md`, y los cuerpos de cada
respuesta archivados por tipo en `capabilities/`, `vectors/`, `features/`, `epochs/` y
`chips/`. Las dos últimas, que son imágenes y se pueden volver a pedir, están ignoradas
por git; el resto se versiona, porque es la auditoría. Esa es la razón de ser del
manifiesto: **una cifra sin su línea en el manifiesto no es una medición**.

Vocabulario mínimo, para que el resto se lea sin tropiezos:

| Término | En una línea |
|---|---|
| **extent** | el rectángulo de estudio: cuatro números (mín. x, mín. y, máx. x, máx. y) |
| **CRS** | el sistema de coordenadas en que están escritos esos números (`EPSG:4326` son grados) |
| **GSD** | cuántos metros de suelo mide un píxel: 0,25 m/px es un píxel por cada 25 cm |
| **época** | una pasada del vuelo fotogramétrico, en la práctica un año de ortofoto |
| **tesela** | el recorte cuadrado fijo con el que el pipeline trocea la imagen |
| **`ABSENT:`** | el servicio no dio el dato, y la línea dice por qué. Es un resultado, no un fallo |

Los servicios se alcanzan desde tu máquina y, a propósito, **no** desde el sandbox de los
agentes: una ejecución ahí dentro reporta ausencia en todo, y eso es correcto.

**Qué mirar:** si ninguna sección llega a hablar con un servicio, toda cifra de abajo es
`ABSENT` y lo que estás midiendo es tu acceso a la red, no el dato.


## 2 · Parámetros: las únicas líneas que se editan

La celda siguiente reúne **todo** aquello de lo que depende esta ejecución, una variable
por línea con su comentario. Cambiar de extent, de proveedor o de rango de años es editar
esta celda y ninguna otra.

Dos valores están **vacíos a propósito**: `EXTENT` y `EXTENT_CRS`. Elegir el extent es el
paso 1 de EXP-001 y sigue abierto, así que aquí no hay ningún rectángulo escrito: o lo
dibujas en el mapa de la sección 3, o pegas sus cuatro números aquí a mano.

Los hosts, los nombres de capa y los atributos que se esperan del catastro **no están en
el notebook**: viven en la entrada `PROVIDERS[PROVIDER_KEY]` de `eda.py`, el único
sitio del proyecto donde se nombra un servicio. El tamaño de píxel también sale de ahí
(`default_gsd_m`), nunca de un número escrito a mano, así que cubrir otro país es añadir
una entrada en vez de tocar este notebook.

**Qué mirar:** que el panel que imprime la celda muestre el proveedor, el rango de años y
la carpeta de salidas que esperabas, y que el extent aparezca como pendiente.


In [7]:
# ═══ Parámetros · las únicas líneas que se editan ═══════════════════════════
EXTENT: tuple[float, ...] = ()  # mín. x, mín. y, máx. x, máx. y — vacío: el paso 1 de EXP-001 sigue abierto
EXTENT_CRS = ""  # el CRS en que están escritos esos cuatro números — vacío hasta elegirlo
PROVIDER_KEY = "es-pnoa"  # qué entrada de PROVIDERS (eda.py) describe los servicios
YEAR_FIRST = 2004  # primer año del rango de épocas que sondea la sección 5
YEAR_LAST = 2024  # último año de ese rango, incluido — 2024 es el último que anuncia el servicio
REQUEST_PX = 512  # ancho y alto, en píxeles, de cada imagen pedida sobre el extent completo
TILE_PX = 256  # tamaño de tesela del pipeline: fija el número de teselas y el tamaño del chip
SNAP_GRID_M = 0.0  # rejilla de ajuste en metros; 0.0 significa TILE_PX × GSD, o sea una tesela entera
N_EXAMPLES = 6  # cuántos edificios de ejemplo llevan tira de chips en la sección 10
OUT_DIR_NAME = "out"  # dónde va todo, al lado de este notebook: el experimento se mueve entero
# ═══ Debajo de esta línea no hay parámetros ════════════════════════════════

from pathlib import Path

from IPython.display import Markdown, display

import eda

PROVIDER = eda.PROVIDERS[PROVIDER_KEY]
GSD_M = PROVIDER.default_gsd_m  # tamaño de píxel: dato del proveedor, nunca un literal
SNAP_M = SNAP_GRID_M or TILE_PX * GSD_M  # la rejilla efectiva a la que se ajusta el extent
YEARS = range(YEAR_FIRST, YEAR_LAST + 1)
HERE = Path(eda.__file__).resolve().parent  # la carpeta del experimento, no el cwd
OUT_DIR = eda.prepare_out(HERE / OUT_DIR_NAME)  # crea out/ y una subcarpeta por tipo


def panel(title: str, rows: list[tuple[str, str]], note: str = "") -> None:
    """Muestra pares etiqueta/valor como una tabla, con una nota opcional debajo.

    Es la forma en que todas las secciones presentan sus resultados: una tabla
    legible en lugar de una ráfaga de `print`.
    """
    table = "\n".join(f"| {label} | {value} |" for label, value in rows)
    tail = f"\n\n{note}" if note else ""
    display(Markdown(f"**{title}**\n\n| | |\n|---|---|\n{table}{tail}"))


extent_row = (
    f"`{tuple(EXTENT)}` en `{EXTENT_CRS or 'CRS sin indicar'}`"
    if EXTENT
    else "_vacío — se elige en la sección 3_"
)
panel(
    "Configuración de esta ejecución",
    [
        ("📍 Proveedor", f"{PROVIDER.label} (`{PROVIDER_KEY}`)"),
        ("🗺️ CRS nativo", f"`{PROVIDER.native_crs}` a {GSD_M} m/px"),
        ("📅 Épocas a sondear", f"{YEAR_FIRST}–{YEAR_LAST} ({len(YEARS)} años)"),
        ("🖼️ Tamaño de petición", f"{REQUEST_PX} px sobre el extent · tesela {TILE_PX} px"),
        ("📐 Rejilla de ajuste", f"{SNAP_M:g} m (= {TILE_PX} px × {GSD_M} m)"),
        ("🏠 Edificios de ejemplo", f"{N_EXAMPLES}"),
        ("💾 Salidas", f"carpeta `{OUT_DIR.relative_to(HERE)}` (imágenes ignoradas por git)"),
        ("⏳ Extent", extent_row),
    ],
    note="Esta celda no pide nada a ningún servicio: solo configura y crea la carpeta de salidas.",
)


**Configuración de esta ejecución**

| | |
|---|---|
| 📍 Proveedor | Spain: PNOA orthophotos and cadastral building footprints (`es-pnoa`) |
| 🗺️ CRS nativo | `EPSG:25830` a 0.25 m/px |
| 📅 Épocas a sondear | 2004–2024 (21 años) |
| 🖼️ Tamaño de petición | 512 px sobre el extent · tesela 256 px |
| 📐 Rejilla de ajuste | 64 m (= 256 px × 0.25 m) |
| 🏠 Edificios de ejemplo | 6 |
| 💾 Salidas | carpeta `out` (imágenes ignoradas por git) |
| ⏳ Extent | _vacío — se elige en la sección 3_ |

Esta celda no pide nada a ningún servicio: solo configura y crea la carpeta de salidas.

### 2.1 · Utilidades compartidas: cómo se anota cada veredicto

Las secciones de abajo **no lanzan excepciones** cuando un servicio no contesta: anotan la
ausencia con su motivo y siguen adelante. Esta celda define las tres funciones que usan
todas, y por eso va antes que cualquier medición:

- `record(sección, línea)` — guarda e imprime la única línea de veredicto de una sección;
  la sección 12 las recoge todas para el resumen final.
- `absent(sección, motivo)` — lo mismo, pero la línea empieza por `ABSENT: ` y dice **por
  qué** no hubo dato.
- `attempt(qué, llamada)` — envuelve una petición que puede fallar *antes* de salir de esta
  máquina (un almacén de certificados sin CA, un proxy mal configurado). Devuelve el mismo
  tipo de valor que una petición normal, con el motivo dentro de `error`.

Nada se oculta: el motivo viaja siempre en el resultado. Eso es lo que permite leer el
notebook de principio a fin como un mapa de lo que hay y lo que falta, en vez de encontrar
un traceback a mitad y no saber cuánto se llegó a medir.

**Qué mirar:** la línea de confirmación al final de la celda. A partir de aquí, cualquier
`ABSENT:` que aparezca abajo viene con su motivo pegado.


In [8]:
# --- Cómo se registra lo que mide cada sección -----------------------------
%matplotlib inline

from collections.abc import Callable

FINDINGS: dict[str, str] = {}  # sección -> su línea de veredicto, recogidas por la sección 12
MEASURED: dict[str, object] = {}  # sección -> las cifras que la sección 12 puede llamar medidas


def record(section: str, line: str) -> None:
    """Guarda la línea de veredicto de una sección y la imprime."""
    FINDINGS[section] = line
    print(line)


def absent(section: str, reason: str) -> None:
    """Anota que una sección no tuvo nada que medir, y por qué."""
    record(section, "ABSENT: " + reason)


def attempt(what: str, call: Callable[[], eda.Fetch]) -> eda.Fetch:
    """Lanza una petición que puede fallar antes incluso de salir de esta máquina.

    `eda.fetch` ya convierte un fallo de red en un valor; esto hace lo mismo con
    un fallo anterior a la red, para que quien llame lo reporte con sus palabras.
    """
    try:
        return call()
    except OSError as exc:
        return eda.Fetch(
            url=f"no enviada: {what}",
            status=0,
            nbytes=0,
            md5="",
            elapsed_s=0.0,
            path=None,
            error=f"{type(exc).__name__}: {exc}",
        )


print("utilidades listas · veredictos: record() / absent() · peticiones: attempt()")
print(f"manifiesto de la ejecucion: {eda.manifest_path(OUT_DIR).relative_to(HERE)}")


utilidades listas · veredictos: record() / absent() · peticiones: attempt()
manifiesto de la ejecucion: out/manifest.csv


## 3 · Elegir el extent dibujándolo en el mapa

El extent es la única entrada que no se deduce de nada, así que se elige a mano. El mapa
abre sobre **imagen de satélite** (basemap Esri) para que nunca dibujes a ciegas, y encima
lleva la **ortofoto del proveedor** como capa conmutable desde el control de capas: esa es
la imagen que se va a medir de verdad, y si sale en negro o con teselas de error, el nombre
de capa de la entrada del proveedor está mal — lo confirma la sección 4. Solo está activa
la herramienta de **rectángulo**; un extent es un rectángulo o no es nada.

Tres pasos, en este orden:

1. **Dibuja un rectángulo** (o reajústalo: la edición está activada). Al soltar el ratón
   verás sus esquinas SO/NE impresas debajo del mapa, y el rectángulo queda guardado en
   `drawing.json` en ese mismo instante.
2. **Pulsa «Usar este rectángulo».** No es decorativo: es la prueba de que el kernel está
   recibiendo lo que haces. Si el texto de al lado cambia, el puente navegador↔kernel
   funciona; si no cambia, está muerto y lo que toca es rellenar `EXTENT` y `EXTENT_CRS` en
   la sección 2.
3. **Ejecuta la celda siguiente**, que lo resuelve y lo mide.

Un par de cientos de metros de lado es el orden de magnitud correcto: pequeño para poder
revisar a mano todos sus edificios en la sección 10, grande para que haya unas decenas.
Mientras dibujas no se pide nada a ningún servicio: las teselas las carga tu navegador.
*Run All* **no puede** funcionar aquí, porque entre estas dos celdas hace falta una persona.

**Qué mirar:** que el texto junto al botón pase a describir tu rectángulo. Eso, y solo eso,
confirma que el valor ha llegado al kernel.


In [10]:
# --- El mapa, el rectángulo y el botón que confirma que llegó al kernel ----
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import (
    DrawControl,
    LayersControl,
    Map,
    MeasureControl,
    ScaleControl,
    WMSLayer,
    basemap_to_tiles,
    basemaps,
)

DRAW_CRS = "EPSG:4326"  # un mapa web devuelve los rectángulos en longitud y latitud
SELECTION: dict[str, object] = {}  # el último rectángulo que ha llegado al kernel
DRAW_EVENTS: list[str] = []  # lo que el kernel ha recibido del mapa: el diagnóstico

view_lat, view_lon, view_zoom = PROVIDER.default_view 
extent_map = Map(
    basemap=basemap_to_tiles(basemaps.Esri.WorldImagery),  # satélite: nunca se dibuja a ciegas
    center=(view_lat, view_lon),
    zoom=view_zoom,
    scroll_wheel_zoom=True,
)

# La ortofoto del proveedor encima del satélite. El endpoint y el nombre de capa salen de
# la entrada PROVIDERS, y las dos búsquedas tienen valor por defecto en vez de reventar:
# un nombre de capa que el servicio no sirve degrada a teselas de error en el navegador.
# Las teselas las pide el navegador, así que esta celda no toca la red.
ortho_service, ortho_layer = PROVIDER.layers.get("ortho", (PROVIDER.epoch_service, ""))
extent_map.add(
    WMSLayer(
        url=PROVIDER.endpoints.get(ortho_service, ""),
        layers=ortho_layer,
        format="image/jpeg",  # una ortofoto es fotográfica y opaca
        transparent=False,
        attribution=PROVIDER.label,
        name=f"ortofoto: {ortho_layer or 'sin capa en la entrada'} (sin confirmar)",
    )
)

draw_control = DrawControl(
    rectangle={"shapeOptions": {"color": "#d95f02", "weight": 2, "fillOpacity": 0.1}},
    polyline={},  # el resto de herramientas, desactivadas: hay una sola forma de usar esto
    polygon={},
    circle={},
    circlemarker={},
    marker={},
    edit=True,  # se puede reajustar el rectángulo arrastrando sus vértices
    remove=True,  # la papelera, para tirar uno del tamaño equivocado
)
coord_label = widgets.Label("Sin rectángulo todavía. Dibuja uno en el mapa.")
use_button = widgets.Button(description="Usar este rectángulo", button_style="success")


def rectangle_bbox(feature: object) -> tuple[float, ...]:
    """Los cuatro números de un rectángulo GeoJSON, o () si eso no es un polígono."""
    geometry = feature.get("geometry") if isinstance(feature, dict) else None
    geometry = geometry or {}
    if geometry.get("type") != "Polygon":
        return ()
    ring = [(float(x), float(y)) for x, y in geometry["coordinates"][0]]
    return (
        min(p[0] for p in ring),
        min(p[1] for p in ring),
        max(p[0] for p in ring),
        max(p[1] for p in ring),
    )


def describe(bbox: tuple[float, ...]) -> str:
    """El rectángulo en las dos esquinas que una persona puede comprobar a ojo."""
    report = eda.extent_report(bbox, DRAW_CRS, gsd_m=GSD_M, tile_px=TILE_PX)
    return (
        f"SO ({bbox[1]:.6f}, {bbox[0]:.6f}) · NE ({bbox[3]:.6f}, {bbox[2]:.6f}) · "
        f"{report['width_m']:.0f} × {report['height_m']:.0f} m · {report['area_km2']:.4f} km²"
    )


def handle_draw(target: object, action: str, geo_json: object, **kwargs: object) -> None:
    """Corre en el kernel al dibujar, editar o borrar. Imprime y guarda en el acto.

    Guardar aquí, y no al resolver, es lo que hace que el rectángulo sobreviva a un
    reinicio del kernel o a reejecutar esta celda. Se imprime en la salida de la celda
    (no en un widget `Output`) porque un fallo dentro de un callback que escribe en un
    widget desaparece sin rastro, y aquí hace falta verlo.
    """
    DRAW_EVENTS.append(action)
    if action == "deleted":
        SELECTION.clear()
        gone = eda.forget_drawing(OUT_DIR)
        coord_label.value = "Rectángulo borrado. Dibuja otro."
        print("rectángulo borrado" + (" · drawing.json eliminado" if gone else ""))
        return
    bbox = rectangle_bbox(geo_json)
    if not bbox:
        print(f"el kernel recibió '{action}', pero sin rectángulo dentro: {geo_json!r}")
        return
    SELECTION["bbox"] = bbox
    SELECTION["crs"] = DRAW_CRS
    saved = eda.save_drawing(bbox, DRAW_CRS, OUT_DIR)
    coord_label.value = describe(bbox)
    print(f"rectángulo {action} · {describe(bbox)}")
    print(f"guardado en {saved.relative_to(HERE)} — pulsa el botón y ejecuta la celda siguiente")


def use_selection(button: object) -> None:
    """Confirma el rectángulo: reescribe `drawing.json` y lo dice en la etiqueta."""
    if not SELECTION.get("bbox"):
        coord_label.value = (
            "Ningún rectángulo en el kernel. Si has dibujado uno y esto no cambia, "
            "el puente widget↔kernel no entrega: usa EXTENT en la sección 2."
        )
        return
    bbox = tuple(SELECTION["bbox"])
    saved = eda.save_drawing(bbox, str(SELECTION["crs"]), OUT_DIR)
    coord_label.value = f"✅ en uso · {describe(bbox)} · guardado en {saved.name}"
    print(f"confirmado: {eda.extent_env_lines(bbox, DRAW_CRS)}")


draw_control.on_draw(handle_draw)
use_button.on_click(use_selection)
extent_map.add(draw_control)
extent_map.add(
    MeasureControl(
        position="bottomleft",
        primary_length_unit="meters",
        secondary_length_unit="kilometers",
        primary_area_unit="sqmeters",
    )
)
extent_map.add(ScaleControl(position="bottomright"))
extent_map.add(LayersControl(position="topright"))  # conmuta ortofoto / satélite

display(extent_map)
display(use_button)
display(coord_label)


Map(center=[40.4, -3.7], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

Button(button_style='success', description='Usar este rectángulo', style=ButtonStyle())

Label(value='Sin rectángulo todavía. Dibuja uno en el mapa.')

### 3.1 · Resolver, medir y registrar el extent

Esta celda decide qué extent usa el resto del notebook y **dice de dónde lo ha sacado**,
porque una cifra medida sobre el extent equivocado es peor que no tener cifra. Busca en
este orden y se queda con el primero que encuentra:

1. el rectángulo que tiene el kernel, el que llegó por el evento del mapa o por el botón;
2. lo que guarda el propio control de dibujo (`draw.data`, `last_draw`), por si el evento
   no llegó pero su estado sí se sincronizó;
3. `EXTENT` y `EXTENT_CRS` de la sección 2, si los has rellenado a mano;
4. `drawing.json`, un rectángulo dibujado antes — **esto es lo que hace que reiniciar el
   kernel no te obligue a volver a dibujar**;
5. nada: imprime `ABSENT: ` con el motivo y un diagnóstico de qué recibió el kernel, las
   secciones siguientes reportan ausencia, y el notebook llega al final sin romperse.

Un rectángulo dibujado en un mapa web llega en longitud y latitud, que no es un sistema en
el que se puedan medir metros, así que se reproyecta al CRS nativo del proveedor como
**envolvente de las cuatro esquinas**: reproyectar solo dos esquinas opuestas perdería
terreno por dos lados. `skew_m` es el precio de haber dibujado en grados —cuánto se separan
en X las dos esquinas occidentales al proyectar— y la envolvente es lo que lo paga; el
tamaño geodésico va al lado como comprobación de cordura.

Luego se ajusta hacia fuera a teselas enteras, y ese extent ajustado es el que se escribe
en `extent.json` y el que se pega en EXP-001. Redibuja y reejecuta las veces que quieras:
sobreescribe en vez de acumular, y **nunca borra** un extent ya registrado.

**Qué mirar:** la fila marcada con ✅ en la tabla de procedencia, y `exact_fit` en el
extent ajustado — `True` significa que la rejilla de teselas lo divide sin resto.


In [11]:
# --- De dónde sale el extent, en orden, y qué mide ------------------------
candidates: list[tuple[str, tuple[float, ...], str]] = []

if SELECTION.get("bbox"):  # 1 · el kernel tiene el rectángulo
    candidates.append(
        ("el rectángulo que tiene el kernel", tuple(SELECTION["bbox"]), str(SELECTION["crs"]))
    )

for feature in reversed(list(draw_control.data)):  # 2 · el estado del control de dibujo
    from_data = rectangle_bbox(feature)
    if from_data:
        candidates.append(("el control de dibujo (data)", from_data, DRAW_CRS))
        break

if draw_control.last_action not in {"", "deleted"}:  # 2b · dibujado y no borrado después
    from_last = rectangle_bbox(draw_control.last_draw)
    if from_last:
        candidates.append(("el control de dibujo (last_draw)", from_last, DRAW_CRS))

if len(EXTENT) == 4 and EXTENT_CRS:  # 3 · escrito a mano en la sección 2
    candidates.append(
        ("el parámetro EXTENT de la sección 2", tuple(float(v) for v in EXTENT), EXTENT_CRS)
    )

previous = eda.load_drawing(OUT_DIR)  # 4 · un dibujo de antes del reinicio
if previous is not None:
    candidates.append((f"drawing.json, dibujado el {previous[2]}", previous[0], previous[1]))

EXTENT_BBOX: tuple[float, ...] = ()
EXTENT_IN = ""

if not candidates:
    panel(
        "⏳ Sin extent todavía",
        [
            ("Eventos recibidos del mapa", f"{DRAW_EVENTS or 'ninguno'}"),
            ("Rectángulos en el control", f"{len(draw_control.data)}"),
            ("Última acción del control", f"`{draw_control.last_action or 'ninguna'}`"),
            ("drawing.json", "no existe"),
            ("EXTENT en la sección 2", "vacío"),
        ],
        note=(
            "Si dibujaste un rectángulo y aquí no hay ningún evento, el puente "
            "navegador↔kernel no está entregando nada: rellena `EXTENT` y `EXTENT_CRS` "
            "en la sección 2 y reejecuta esta celda."
        ),
    )
    absent(
        "3 extent",
        "ningún extent seleccionado — dibuja un rectángulo arriba, o rellena EXTENT y "
        "EXTENT_CRS en la sección 2",
    )
    stale = OUT_DIR / "extent.json"
    if stale.is_file():
        print(f"nota: {stale.name} de una ejecución anterior sigue en disco, y no se usa aquí")
else:
    source, bbox, bbox_crs = candidates[0]
    panel(
        "De dónde sale el extent",
        [
            ("✅ usado" if n == 0 else "· disponible", name)
            for n, (name, _, _) in enumerate(candidates)
        ],
        note="Se usa el primero de la lista; los demás quedan como respaldo.",
    )
    geographic = eda.to_projected(bbox, bbox_crs, DRAW_CRS)
    native = eda.to_projected(bbox, bbox_crs, PROVIDER.native_crs)
    as_chosen = eda.extent_report(bbox, bbox_crs, gsd_m=GSD_M, tile_px=TILE_PX)
    EXTENT_BBOX = eda.snap_outward(native, SNAP_M)
    EXTENT_IN = PROVIDER.native_crs
    snapped = eda.extent_report(EXTENT_BBOX, EXTENT_IN, gsd_m=GSD_M, tile_px=TILE_PX)
    saved = eda.save_extent(EXTENT_BBOX, EXTENT_IN, OUT_DIR, gsd_m=GSD_M, tile_px=TILE_PX)
    panel(
        f"El extent elegido, medido · {source}",
        [
            ("🧭 Tal cual, en grados", f"`{tuple(round(v, 6) for v in geographic)}`"),
            ("📦 Envolvente en el CRS nativo", f"`{tuple(round(v, 2) for v in native)}`"),
            ("📏 Tamaño", f"{as_chosen['width_m']:.1f} × {as_chosen['height_m']:.1f} m"),
            ("🧮 Área", f"{as_chosen['area_km2']:.4f} km²"),
            (
                "🌍 Tamaño geodésico",
                f"{as_chosen['geodesic_width_m']:.1f} × {as_chosen['geodesic_height_m']:.1f} m"
                " (comprobación de cordura)",
            ),
            (
                "↘️ skew_m",
                f"{as_chosen['skew_m']:.2f} m — lo que se separan en X las dos esquinas"
                " occidentales al proyectar; la envolvente lo absorbe, no se pierde terreno",
            ),
            (
                "🧱 Ajustado a la rejilla",
                f"`{tuple(round(v, 2) for v in EXTENT_BBOX)}` ({SNAP_M:g} m)",
            ),
            ("🖼️ Píxeles", f"{snapped['width_px']} × {snapped['height_px']} px a {GSD_M} m/px"),
            (
                "🧩 Teselas",
                f"{snapped['tiles'][0]} × {snapped['tiles'][1]} de {TILE_PX} px ·"
                f" exact_fit={snapped['exact_fit']}",
            ),
            ("💾 Registrado en", f"`{saved.relative_to(HERE)}`"),
        ],
    )
    print("pegar en EXP-001 -> Configuration:")
    print(eda.extent_env_lines(EXTENT_BBOX, EXTENT_IN))
    MEASURED["extent"] = {"bbox": list(EXTENT_BBOX), "crs": EXTENT_IN, "report": snapped}
    record(
        "3 extent",
        f"extent desde {source}: {snapped['width_px']}x{snapped['height_px']} px en "
        f"{EXTENT_IN}, {snapped['tiles'][0]}x{snapped['tiles'][1]} teselas, "
        f"guardado como {saved.name}",
    )


**De dónde sale el extent**

| | |
|---|---|
| ✅ usado | drawing.json, dibujado el 2026-11-04T19:18:23+00:00 |

Se usa el primero de la lista; los demás quedan como respaldo.

**El extent elegido, medido · drawing.json, dibujado el 2026-11-04T19:18:23+00:00**

| | |
|---|---|
| 🧭 Tal cual, en grados | `(-3.364126, 40.497182, -3.348985, 40.504393)` |
| 📦 Envolvente en el CRS nativo | `(469144.88, 4483000.42, 470431.06, 4483806.03)` |
| 📏 Tamaño | 1286.2 × 805.6 m |
| 🧮 Área | 1.0362 km² |
| 🌍 Tamaño geodésico | 1283.5 × 800.7 m (comprobación de cordura) |
| ↘️ skew_m | 3.30 m — lo que se separan en X las dos esquinas occidentales al proyectar; la envolvente lo absorbe, no se pierde terreno |
| 🧱 Ajustado a la rejilla | `(469120.0, 4482944.0, 470464.0, 4483840.0)` (64 m) |
| 🖼️ Píxeles | 5376 × 3584 px a 0.25 m/px |
| 🧩 Teselas | 21 × 14 de 256 px · exact_fit=True |
| 💾 Registrado en | `out/extent.json` |

pegar en EXP-001 -> Configuration:
GEO_DECAY_BBOX=469120.0,4482944.0,470464.0,4483840.0
GEO_DECAY_BBOX_CRS=EPSG:25830
extent desde drawing.json, dibujado el 2026-11-04T19:18:23+00:00: 5376x3584 px en EPSG:25830, 21x14 teselas, guardado como extent.json


## 4 · Capacidades: confirmar o contradecir la entrada del proveedor

Antes de medir nada **a través** de la entrada del proveedor hay que ver si lo que afirma
existe. Esta celda pide a cada servicio su documento de capacidades —la lista de lo que
sirve— y lo contrasta afirmación por afirmación: cada CRS, cada nombre de capa, el patrón
de épocas, la stored query del WFS y los atributos esperados del catastro.

Cada fila sale con su veredicto: ✅ `ok` si el servicio lo anuncia, ⚠️ `unverified` si
desde aquí no se puede saber, ❌ `MISMATCH` si dice otra cosa — y entonces la fila trae los
nombres más parecidos que sí anuncia. **Un valor equivocado se reporta, no se arregla
aquí.** Es además la sección más barata de ejecutar, y la que decide si merece la pena
sondear 22 épocas a través de esta entrada.

**Qué mirar:** la columna `veredicto`. Un ❌ en la capa de ortofoto o en el patrón de
épocas invalida todo lo que venga después; uno en el WFS afecta solo a la sección 8.


In [13]:
# --- Sección 4: lo que cada servicio anuncia, contra lo que la entrada afirma ---
import difflib

import pandas as pd

CAPS: dict[str, eda.Capabilities] = {}  # servicio -> lo que anuncia
STORED: tuple[str, ...] = ()  # las stored queries que ofrece el WFS

for service in PROVIDER.endpoints:
    if service == PROVIDER.wfs_service:
        got = attempt(service, lambda: eda.wfs_capabilities(PROVIDER, OUT_DIR))
    else:
        got = attempt(service, lambda s=service: eda.wms_capabilities(PROVIDER, s, OUT_DIR))
    caps = eda.read_capabilities(got.path)
    CAPS[service] = caps
    reason = got.error or caps.exception
    if reason:
        absent(f"4 {service}", f"{service} — {reason}")
    else:
        record(
            f"4 {service}",
            f"{service}: {got.nbytes} bytes, md5 {got.md5[:8]}, "
            f"{len(caps.crs)} CRS, {len(caps.layers)} capas anunciadas",
        )
    if service == PROVIDER.wfs_service:
        listed = attempt(
            f"{service} ListStoredQueries", lambda: eda.wfs_stored_queries(PROVIDER, OUT_DIR)
        )
        STORED = eda.read_capabilities(listed.path).stored_queries
        if not STORED:
            absent(
                f"4 {service} stored queries",
                f"{service} ListStoredQueries — "
                f"{listed.error or 'el servicio no anuncia ninguna stored query'}",
            )

ADVERTISED = tuple(dict.fromkeys(code for caps in CAPS.values() for code in caps.crs))


def judge(claim: str, advertised: tuple[str, ...], asked: bool) -> tuple[str, str]:
    """Veredicto de una afirmación de la entrada contra lo que el servicio anuncia."""
    if not asked:
        return "⚠️ unverified", "el servicio no contestó, así que no se pudo comprobar"
    if claim in advertised:
        return "✅ ok", "anunciado"
    near = difflib.get_close_matches(claim, advertised, n=3, cutoff=0.3)
    tail = f"; más cercanos: {', '.join(near)}" if near else ""
    return "❌ MISMATCH", "no anunciado" + tail


def claim_row(what: str, claim: str, verdict: tuple[str, str]) -> dict[str, str]:
    """Una fila de la tabla: qué se afirma, quién lo afirma y qué contesta el servicio."""
    return {
        "qué": what,
        "la entrada afirma": claim,
        "veredicto": verdict[0],
        "el servicio dice": verdict[1],
    }


claims = [
    claim_row("CRS anunciado", code, judge(code, ADVERTISED, bool(ADVERTISED)))
    for code in PROVIDER.advertised_crs
]

for logical, (service, name) in PROVIDER.layers.items():
    served = CAPS.get(service, eda.read_capabilities(None)).layers
    claims.append(
        claim_row(f"capa '{logical}' en {service}", name, judge(name, served, bool(served)))
    )

epoch_layers = CAPS.get(PROVIDER.epoch_service, eda.read_capabilities(None)).layers
expected = [PROVIDER.epoch_layer_pattern.format(year=year) for year in YEARS]
present = [name for name in expected if name in epoch_layers]
# El patrón es correcto si el servicio anuncia al menos un año con él; lo que importa
# medir es *qué* años, porque ese es el rango real que puede sondear la sección 5.
served_years = [
    year for year in YEARS if PROVIDER.epoch_layer_pattern.format(year=year) in epoch_layers
]
if not epoch_layers:
    epoch_verdict = ("⚠️ unverified", "el servicio de épocas no contestó")
elif not served_years:
    epoch_verdict = ("❌ MISMATCH", "ningún año del rango está anunciado con este patrón")
else:
    detail = f"{len(served_years)} de {len(expected)}: {served_years[0]}–{served_years[-1]}"
    gap = [year for year in YEARS if year not in served_years]
    if gap:
        shown = ", ".join(str(year) for year in gap[:6]) + (" …" if len(gap) > 6 else "")
        detail += f"; sin anunciar: {shown}"
    epoch_verdict = ("✅ ok", detail)
claims.append(
    claim_row(
        f"patrón de épocas en {PROVIDER.epoch_service}",
        f"{PROVIDER.epoch_layer_pattern} en {len(YEARS)} años",
        epoch_verdict,
    )
)

for key, value in PROVIDER.wfs_params.items():
    if "storedquer" in key.lower():  # el nombre del parámetro es cosa de cada servicio
        claims.append(
            claim_row(f"stored query ({key})", value, judge(value, STORED, bool(STORED)))
        )

for logical, name in PROVIDER.attributes.items():
    claims.append(
        claim_row(
            f"atributo '{logical}'",
            name,
            ("⚠️ unverified", "llega con las features, no con las capacidades: sección 8"),
        )
    )

if EXTENT_IN:
    claims.append(
        claim_row("CRS de las peticiones", EXTENT_IN, judge(EXTENT_IN, ADVERTISED, bool(ADVERTISED)))
    )
else:
    absent("4 CRS de las peticiones", "sin extent no hay CRS de peticiones que comprobar")

display(pd.DataFrame(claims))
mismatches = [entry for entry in claims if "MISMATCH" in entry["veredicto"]]
answered = [name for name, caps in CAPS.items() if caps.crs or caps.layers]
panel(
    "Resumen de la sección 4",
    [
        ("📄 Servicios que contestaron", f"{len(answered)} de {len(PROVIDER.endpoints)}"),
        ("🗺️ CRS anunciados (unión)", f"{len(ADVERTISED)}"),
        ("🧱 Capas anunciadas", f"{sum(len(caps.layers) for caps in CAPS.values())}"),
        ("🔎 Stored queries", f"{len(STORED)}"),
        ("❌ Contradicciones", f"{len(mismatches)}"),
    ],
    note="Lo contradicho se corrige en la entrada del proveedor de `eda.py`, no en el notebook.",
)
MEASURED["capabilities"] = {
    "advertised_crs": list(ADVERTISED),
    "stored_queries": list(STORED),
    "claims": claims,
}
record(
    "4 capacidades",
    f"capacidades: contestaron {len(answered)}/{len(PROVIDER.endpoints)} servicios, "
    f"{len(ADVERTISED)} CRS y {sum(len(caps.layers) for caps in CAPS.values())} capas "
    f"anunciadas, {len(mismatches)} contradicciones contra la entrada",
)


ortho: 27902 bytes, md5 cb3fcd0f, 20 CRS, 3 capas anunciadas
ortho_history: 113091 bytes, md5 40b41018, 18 CRS, 37 capas anunciadas
cadastre_wms: 28799 bytes, md5 745793a5, 17 CRS, 13 capas anunciadas
buildings_wfs: 16182 bytes, md5 36610fcf, 11 CRS, 3 capas anunciadas


,qué,la entrada afirma,veredicto,el servicio dice
0,CRS anunciado,EPSG:25829,✅ ok,anunciado
1,CRS anunciado,EPSG:25830,✅ ok,anunciado
2,CRS anunciado,EPSG:25831,✅ ok,anunciado
3,CRS anunciado,EPSG:4326,✅ ok,anunciado
4,capa 'ortho' en ortho,OI.OrthoimageCoverage,✅ ok,anunciado
5,capa 'buildings' en cadastre_wms,CONSTRU,✅ ok,anunciado
6,patrón de épocas en ortho_history,PNOA{year} en 21 años,✅ ok,21 de 21: 2004–2024
7,stored query (STOREDQUERIE_ID),GetBuildingByBBOX,❌ MISMATCH,no anunciado; más cercanos: GetBuildingByParce...
8,atributo 'identifier',localId,⚠️ unverified,"llega con las features, no con las capacidades..."
9,atributo 'condition',conditionOfConstruction,⚠️ unverified,"llega con las features, no con las capacidades..."


**Resumen de la sección 4**

| | |
|---|---|
| 📄 Servicios que contestaron | 4 de 4 |
| 🗺️ CRS anunciados (unión) | 26 |
| 🧱 Capas anunciadas | 56 |
| 🔎 Stored queries | 5 |
| ❌ Contradicciones | 1 |

Lo contradicho se corrige en la entrada del proveedor de `eda.py`, no en el notebook.

capacidades: contestaron 4/4 servicios, 26 CRS y 56 capas anunciadas, 1 contradicciones contra la entrada


## 5 · Épocas: qué años contestan y qué resolución declara cada uno

Esta sección pide una imagen pequeña por cada año del rango y anota, fila por año, lo que
contesta el servicio: eso se mide pidiendo, no leyendo el catálogo.

La resolución es otra pregunta y va **siempre con su fuente**. No hay campo de escala: se
declara en prosa, en el `Abstract` de cada capa de año, y **por región**, y se leen todas
con sus palabras. Un año sin declaración sale como `no declarada`, y **nunca hereda** el
número del vecino, de la capa padre ni el `default_gsd_m` de la entrada. Aparte va la
resolución **pedida** al sondear: lo que pedimos, no lo que voló el sensor.

Ninguna es una medición: medir exige pedir el mismo bbox a más píxeles hasta que el detalle
deje de crecer. Las capacidades tampoco dicen en qué región cae el extent; eso lo contesta
su capa de cobertura por `GetFeatureInfo`, en otro paso.

**Qué mirar:** la columna `resolución declarada` junto a su `fuente`: un número sin fuente
invalida la sección.

In [15]:
# --- Sección 5: qué años contestan sobre el extent, y qué declara cada uno ---
import pandas as pd

EPOCH_CAPS = CAPS.get(PROVIDER.epoch_service, eda.read_capabilities(None))
NOT_ASKED = "no se pidió: la sección 3 no resolvió ningún extent"
EPOCH_ROWS: list[dict[str, object]] = []  # una fila por año, como contestó el servicio
DECLARED: dict[int, tuple[eda.DeclaredResolution, ...]] = {}  # año -> lo declarado

if EXTENT_BBOX:
    EPOCH_ROWS = eda.epoch_availability(
        PROVIDER, EXTENT_BBOX, EXTENT_IN, YEARS, OUT_DIR
    )
    ground = eda.extent_report(EXTENT_BBOX, EXTENT_IN, gsd_m=GSD_M, tile_px=TILE_PX)
    # Lo que PEDIMOS: metros de suelo por píxel de la imagen de sondeo. No es la
    # resolución del vuelo, y el tamaño de petición lo fija eda, no esta celda.
    asked = (
        f"{float(ground['width_m']) / eda.PROBE_SIZE[0]:.2f} × "
        f"{float(ground['height_m']) / eda.PROBE_SIZE[1]:.2f} m/px"
        f" ({eda.PROBE_SIZE[0]}×{eda.PROBE_SIZE[1]} px sobre el extent completo)"
    )
else:
    asked = NOT_ASKED
probed = {int(row["year"]): row for row in EPOCH_ROWS}

epoch_table: list[dict[str, object]] = []
for year in YEARS:
    layer = PROVIDER.epoch_layer_pattern.format(year=year)
    advertised = layer in EPOCH_CAPS.layers
    found = eda.declared_resolutions(EPOCH_CAPS.abstracts.get(layer, ""))
    DECLARED[year] = found
    if found:  # tal cual lo declara el servicio: una cifra por región nombrada
        declared = " · ".join(f"{d.gsd_m:g} m" for d in found)
        origin = f"`Abstract` de la capa, {len(found)} declaración(es)"
    elif EPOCH_CAPS.exception:  # catálogo no leído: no se hereda nada de nadie
        declared, origin = "no declarada", f"catálogo: {EPOCH_CAPS.exception}"
    elif not advertised:
        declared, origin = "no declarada", "la capa no está anunciada en el catálogo"
    else:
        declared, origin = "no declarada", "su `Abstract` no declara resolución"
    got = probed.get(year)
    verdict = str(got["verdict"]) if got else "unknown"
    evidence = str(got["evidence"]) if got else NOT_ASKED
    epoch_table.append(
        {
            "año": year,
            "anunciada": "sí" if advertised else "no",
            "veredicto": verdict,
            "evidencia": evidence,
            "blanco": got["blank"] if got else None,
            "estado": int(got["status"]) if got else 0,
            "bytes": int(got["nbytes"]) if got else 0,
            "md5": str(got["md5"])[:8] if got else "",
            "resolución declarada": declared,
            "fuente de la resolución": origin,
        }
    )
    if verdict != "present":  # una línea por año, con su motivo completo
        print(
            f"ABSENT: {year} — sin imagen ({verdict}: {evidence});"
            f" resolución {declared} ({origin})"
        )

display(pd.DataFrame(epoch_table))

quotes = [
    (f"{year} · {d.gsd_m:g} m", f"«{d.text}»")
    for year, found in DECLARED.items()
    for d in found
]
if quotes:
    panel(
        "Resolución declarada, con las palabras del servicio",
        quotes,
        note=(
            "Cada fila es una declaración, no una medición, y la cita es la región tal "
            "como la nombra el servicio. Las capacidades no dicen en qué región cae "
            "este extent: lo contesta su capa de cobertura por `GetFeatureInfo`."
        ),
    )

present_years = [
    int(row["año"]) for row in epoch_table if row["veredicto"] == "present"
]
declaring = [year for year, found in DECLARED.items() if found]
distinct = sorted({d.gsd_m for found in DECLARED.values() for d in found})
panel(
    "Resumen de la sección 5",
    [
        ("📅 Años del rango", f"{len(epoch_table)} ({YEAR_FIRST}–{YEAR_LAST})"),
        (
            "🧾 Anunciados por el servicio",
            f"{sum(1 for row in epoch_table if row['anunciada'] == 'sí')}",
        ),
        ("🖼️ Con imagen sobre el extent", f"{len(present_years)}"),
        (
            "📐 Declaran resolución",
            f"{len(declaring)} años · valores "
            f"{', '.join(f'{v:g}' for v in distinct) or '—'} m",
        ),
        (
            "📨 Resolución pedida al sondear",
            f"{asked} — lo que pedimos, no lo que voló el sensor",
        ),
        (
            "📌 `default_gsd_m` de la entrada",
            f"{GSD_M} m/px — supuesto para la aritmética de teselas, no una medición",
        ),
    ],
    note=(
        "Declarada ≠ medida: medir la resolución nativa exige pedir el mismo bbox a "
        "más píxeles hasta que el detalle deje de crecer, y eso no es de esta sección."
    ),
)
MEASURED["epochs"] = {
    "probed": epoch_table,
    "present_years": present_years,
    "requested_gsd": asked,
    # declarado, no medido: la sección 12 no puede presentarlo como una medición
    "declared_gsd_m_not_measured": {
        year: [d.gsd_m for d in found] for year, found in DECLARED.items() if found
    },
}
if present_years:
    record(
        "5 épocas",
        f"épocas: {len(present_years)} de {len(epoch_table)} años con imagen sobre "
        f"el extent ({present_years[0]}–{present_years[-1]}), {len(declaring)} con "
        f"resolución declarada en su Abstract; pedido a {asked}",
    )
else:
    reasons = sorted({str(row["evidencia"]) for row in epoch_table})
    absent(
        "5 épocas",
        f"ningún año del rango devolvió imagen sobre el extent — {len(epoch_table)} "
        f"sondeos, {len(reasons)} motivo(s) distinto(s), el primero «{reasons[0]}»",
    )

ABSENT: 2004 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2005 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m · 0.4 m (`Abstract` de la capa, 3 declaración(es))
ABSENT: 2007 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2008 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2010 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2013 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m (`Abstract` de la capa, 1 declaración(es))
ABSENT: 2016 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.

,año,anunciada,veredicto,evidencia,blanco,estado,bytes,md5,resolución declarada,fuente de la resolución
0,2004,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
1,2005,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m · 0.4 m,"`Abstract` de la capa, 3 declaración(es)"
2,2006,sí,present,image with pixel content,False,200,195007,9817c876,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
3,2007,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
4,2008,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
5,2009,sí,present,image with pixel content,False,200,200498,eacd3ca7,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
6,2010,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
7,2011,sí,present,image with pixel content,False,200,199029,503520ad,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
8,2012,sí,present,image with pixel content,False,200,52282,a673e0a8,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
9,2013,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m,"`Abstract` de la capa, 1 declaración(es)"


**Resolución declarada, con las palabras del servicio**

| | |
|---|---|
| 2004 · 0.5 m | «Abarca las Comunidades de Andalucía, Castilla y León, Cataluña, Comunitat Valenciana, Galicia y Murcia con resolución de 0,5m» |
| 2004 · 0.25 m | «y País Vasco y La Rioja con resolución de 0,25 m» |
| 2005 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía, Cantabria, SW de Castilla y León, Cataluña, Comunitat Valenciana, Extremadura, Galicia y País Vasco con una resolución de 0,5 m» |
| 2005 · 0.25 m | «y la parte NE de Castilla y León con resolución de 0,25 m» |
| 2005 · 0.4 m | «y Canarias con resolución de 0,40 m» |
| 2006 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía, Aragón, Isalas Baleares, Castilla-La Mancha, NW de CAstilla y León, Cataluña, Extremadura, La Rioja, Madrid, Comunitat Valenciana y las ciudades de Ceuta y Melilla con una resolución de 0,5 m» |
| 2006 · 0.25 m | «y País Vasco, Navarra, SE de Castilla y León y Asturias con una resolución de 0,25 m» |
| 2007 · 0.5 m | «Abarca las Comunidades Autónomas de N de Andalucía, Norte de Castilla y León, Cataluña, SW de Extremadura, País Vasco y Comunitat Valenciana con una resolución de 0,5 m» |
| 2007 · 0.25 m | «y Cantabria, SW de Castilla y León, Se de Extremadura, Galicia y Murcia con resolución de 0,25 m» |
| 2008 · 0.5 m | «Abarca las Comunidades Autónomas de Sur de Andalucía, Baleares, SE de Castilla y León, Cataluña y Comunitat Valenciana a una resolución de 0,50m» |
| 2008 · 0.25 m | «y NW de Castilla y León, Capitales de provincia de Cataluña, Extremadura, Galicia, Navarra y País Vasco a una resolución de 0.25 m» |
| 2009 · 0.5 m | «Abarca las Comunidades Autónomas de Norte de Andalucía, Aragón, SW de Castilla y León, SW y NE de Extremadura y Madrid a una resolución de 0,50m» |
| 2009 · 0.25 m | «y la Comunitat Valenciana, Murcia, La Rioja, Galicia, SE de Extremadura, Cataluña, NE de Castilla y León, Castilla-La Mancha, Islas Canarias y Asturias a una resolución de 0,25 m» |
| 2010 · 0.5 m | «Abarca las Comunidades Autónomas de Sur de Andalucía, NW de Castilla y León, NW de Extremadura a una resolución de 0,50m» |
| 2010 · 0.25 m | «y Las Illes Balears, Cantabria, SE de Castilla y León, Cataluña, NE de Extremadura, Galicia, País Vasco y Comunitat Valenciana a una resolución de 0.25 m» |
| 2011 · 0.5 m | «Abarca las Comunidades Autónomas de Norte de Andalucía, Norte de Castilla y León, SE de Extremadura, y Murcia a una resolución de 0,50 m» |
| 2011 · 0.25 m | «y Sur de Castilla y León, Cataluña, SW de Extremadura, Galicia, Madrid y Asturias a una resolución de 0,25 m» |
| 2012 · 0.5 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, interior de Cataluña, NE de Extremadura, La Rioja y Comunitat Valenciana a una resolución de 0,50 m» |
| 2012 · 0.25 m | «y Las Illes Balears, las Islas Canarias, la costa de Cataluña, Navarra, País Vasco y el NW de Extremadura a una resolución de 0,25 m» |
| 2013 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía, Extremadura y Murcia a una resolución de 0,50 m» |
| 2014 · 0.25 m | «Abarca las Comunidades Autónomas de Asturias, Cantabria, Galicia, La Rioja, Madrid, País Vasco y Alicante a una resolución de 0,25 m» |
| 2014 · 0.5 m | «, Castilla y León y Navarra a una resolución de 0.50 m» |
| 2014 · 0.1 m | «y las ciudades de Ceuta y Melilla a una resolución de 0.10 m» |
| 2015 · 0.5 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, interior de Cataluña a una resolución de 0,50 m» |
| 2015 · 0.25 m | «y Comunitat Valenciana, Illes Balears, las islas Canarias y la costa de Cataluña a una resolución de 0,25 m» |
| 2016 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía a una resolución de 0,50 m» |
| 2016 · 0.25 m | «y Extremadura y Murcia a una resolución de 0,25 m» |
| 2017 · 0.25 m | «Abarca las Comunidades Autónomas de Asturias, Cantabria, Castilla y León, Galicia, La Rioja, Madrid, Navarra, País Vasco y Alicante a una resolución de 0,25 m» |
| 2018 · 0.25 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, Cataluña y Comunitat Valenciana a una resolución de 0,25 m» |
| 2018 · 0.15 m | «y las Illes Balears y las islas Canarias a una resolución de 0,15 y 0,16 m» |
| 2018 · 0.16 m | «y las Illes Balears y las islas Canarias a una resolución de 0,15 y 0,16 m» |
| 2019 · 0.25 m | «Abarca las Comunidades Autónomas de Andalucía, Extremadura y Región de Murcia a una resolución de 0,25 m» |
| 2020 · 0.25 m | «Abarca las Comunidades Autónomas de Asturias, Castilla y León, La Rioja, Madrid, País Vasco y Comunidad Foral de Navarra a una resolución de 0,25 m» |
| 2020 · 0.15 m | «y Galicia y Cantabria a 0,15 m» |
| 2021 · 0.25 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, Cataluña, Comunitat Valenciana y las Illes Balears a una resolución de 0,25 m» |
| 2021 · 0.15 m | «y las Islas Canarias a 0,15 m» |
| 2022 · 0.25 m | «Abarca las Comunidades Autónomas de Extremadura, Andalucía y Región de Murcia a una resolución de 0,25 m» |
| 2023 · 0.25 m | «Abarca las Comunidades Autónomas de Castilla y León y Comunidad de Madrid con una resolución de 25 cm» |
| 2023 · 0.15 m | «y La Rioja, Cantabria, País Vasco y Galicia este con una resolución de 15 cm» |
| 2024 · 0.25 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, Cataluña, Comunitat Valenciana y las Illes Balears con una resolución de 25 cm» |
| 2024 · 0.2 m | «y las Islas Canarias con una resolución de 20 cm» |

Cada fila es una declaración, no una medición, y la cita es la región tal como la nombra el servicio. Las capacidades no dicen en qué región cae este extent: lo contesta su capa de cobertura por `GetFeatureInfo`.

**Resumen de la sección 5**

| | |
|---|---|
| 📅 Años del rango | 21 (2004–2024) |
| 🧾 Anunciados por el servicio | 21 |
| 🖼️ Con imagen sobre el extent | 9 |
| 📐 Declaran resolución | 21 años · valores 0.1, 0.15, 0.16, 0.2, 0.25, 0.4, 0.5 m |
| 📨 Resolución pedida al sondear | 5.25 × 3.50 m/px (256×256 px sobre el extent completo) — lo que pedimos, no lo que voló el sensor |
| 📌 `default_gsd_m` de la entrada | 0.25 m/px — supuesto para la aritmética de teselas, no una medición |

Declarada ≠ medida: medir la resolución nativa exige pedir el mismo bbox a más píxeles hasta que el detalle deje de crecer, y eso no es de esta sección.

épocas: 9 de 21 años con imagen sobre el extent (2006–2023), 21 con resolución declarada en su Abstract; pedido a 5.25 × 3.50 m/px (256×256 px sobre el extent completo)
